[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/ab%20db/antibody_db_guide/07_interface/07_interface_lab.ipynb)


# 07 — 항원-항체 interface (1A14 직접 다운로드)

> 본문: [`07_interface.md`](07_interface.md) 와 **한 절씩 짝지어** 보세요.
> **전 셀 실행 10초** (실측 — 도구가 도는 시간만. clone·pip·apt 설치 시간은 빼고 잰 값이에요)
> 코랩 무료 런타임은 코어가 2개뿐이라 설치에 1~2분, 무거운 예측 단계는 몇 배까지 더 걸릴 수 있어요 — 정상입니다.

**이 노트북은 도구를 직접 돌립니다.** RCSB 에서 1A14 를 **직접 내려받아** contact 을 계산하고(`my_run/`), 커밋된 결과와 대조해요.
내 결과는 `my_run/` 에 쌓이고 커밋된 `data/` 는 대조군이에요 — 어느 단계가 실패해도 `resolve()` 가 `data/` 로 폴백해 뒤 절이 계속 돌아갑니다.

## 0) 부트스트랩 — 저장소 클론 · 도구 설치 · 작업 폴더 이동

Colab 은 이 셀 하나로 끝나고, 로컬은 챕터 폴더 안에서 열었다면 클론 없이 진행됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "07_interface"
PIP_PKGS = "pandas matplotlib biopython requests py3Dmol gemmi"          # 이 챕터가 실제로 돌리는 도구 (pip 이름)
NEED_HMMER = False    # ANARCI 계열은 hmmscan(HMMER) 실행파일이 필요해요 (pip 로는 안 깔려요)

import os, sys, subprocess, pathlib, shutil, importlib.util
IN_COLAB = "google.colab" in sys.modules

def _run(cmd, quiet=False):
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        subprocess.run(cmd, shell=True, check=True); return
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        raise subprocess.CalledProcessError(p.returncode, cmd)

_MARK = "antibody_viz.py"           # 이 파일이 있는 폴더가 가이드 루트

def _find_root():
    """가이드 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후엔 cwd 아래만 깊이 3까지 — 위로 올라가 rglob 하면 Colab 에서 / 전체를 뒤집니다.
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "repo 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)        # 챕터 폴더로 이동 → data/·my_run/ 상대경로 동작
sys.path.insert(0, str(ADV_ROOT))   # antibody_viz import 보장
PY, SCRIPTS = sys.executable, ADV_ROOT / "scripts"

# --- 의존성 설치 -----------------------------------------------------------
_IMPORT = {"biopython": "Bio", "pyyaml": "yaml"}          # pip 이름 ≠ import 이름
def _have(pkg):
    mod = _IMPORT.get(pkg, pkg.split("==")[0])
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False
_APT = []                                # 필요한 시스템 패키지를 모아 apt 를 한 번만 돌립니다

_miss = [p for p in PIP_PKGS.split() if not _have(p)]
if _miss:
    _run(f'"{sys.executable}" -m pip -q install ' + " ".join(_miss))

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    _APT.append("fonts-nanum")           # Colab 엔 한글 폰트가 없어 라벨이 □ 로 깨집니다

if _APT:
    _run("apt-get -qq update", quiet=True)   # 인덱스가 낡으면 install 이 404 로 죽습니다
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y " + " ".join(_APT), quiet=True)

# --- 내 산출물 폴더 & 폴백 규칙 --------------------------------------------
MYRUN = pathlib.Path("my_run"); MYRUN.mkdir(exist_ok=True)

QUIET_WARNINGS = True   # 라이브러리 내부 경고 소음을 끕니다. 다 보고 싶으면 False 로 두세요

def run_tool(args, timeout=1800):
    """도구를 서브프로세스로 실제 실행하고 출력을 셀에 그대로 보여줘요.

    igfold·biopython·transformers 는 **자기 패키지 소스 줄번호**가 찍힌
    DeprecationWarning/FutureWarning 을 실제 결과보다 길게 쏟아내요. 그게 성공 메시지를
    덮어버려 실패로 오해하게 만들어서, 기본으로 PYTHONWARNINGS=ignore 를 걸어 둡니다.
    도구가 직접 print 하는 안내·에러는 그대로 남아요(warnings 모듈만 막는 거예요)."""
    args = [str(a) for a in args]
    # 절대경로를 그대로 찍으면 한 줄이 화면을 넘겨 정작 중요한 옵션이 안 보여요.
    # /usr/bin/python3 → python, /…/scripts/x.py → scripts/x.py 로 줄여서 보여줍니다.
    def _short(s):
        if s == PY:
            return "python"
        return "scripts/" + s[len(str(SCRIPTS)) + 1:] if s.startswith(str(SCRIPTS) + os.sep) else s
    print("$", " ".join(_short(a) for a in args))
    env = {**os.environ, "PYTHONWARNINGS": "ignore"} if QUIET_WARNINGS else None
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout, env=env)
    except Exception as e:
        print(f"[실행 실패] {type(e).__name__}: {e}\n→ 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
        return False
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    print(out[-3000:] if out else "(출력 없음)")
    if p.returncode != 0:
        print(f"[실패] returncode={p.returncode} → 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
    return p.returncode == 0

def resolve(name):
    """내가 방금 만든 my_run/ 결과를 우선 쓰고, 없으면 커밋된 data/ 로 폴백."""
    mine, ref = MYRUN/name, pathlib.Path("data")/name
    if mine.exists():
        print(f"[내 결과]   {mine}")
        return str(mine)
    print(f"[레퍼런스] {ref}   ← my_run 산출물이 없어 커밋본으로 이어갑니다")
    return str(ref)

print("작업 폴더 :", pathlib.Path.cwd(), "| Colab:", IN_COLAB)

## 1) 직접 실행 — 복합체 다운로드 + contact 계산 (본문 7.2)

```bash
python scripts/pdb_contacts.py --pdb 1A14 --outdir my_run/pdb                       # ① 사슬 목록 확인
python scripts/pdb_contacts.py --pdb 1A14 --chain1 H --chain2 N --cutoff 4.0 \
    --outdir my_run/pdb --out my_run/contacts_H_N.tsv                                # ② 항원-항체
```
① 을 먼저 돌려 **사슬 ID 를 눈으로 확인**하는 게 핵심이에요 — 1A14 의 항원(neuraminidase)은 chain **N** 이에요.
`--chain2 L` 로 바꾸면 항원이 아니라 **VH–VL packing** 을 보게 됩니다(전혀 다른 분석).
네트워크가 막히면 `--fallback-cif` 로 커밋된 `data/pdb/1A14.cif` 를 씁니다(오프라인 대비).

In [ ]:
import pathlib, pandas as pd
from IPython.display import display

FALLBACK = ["--fallback-cif", "data/pdb/1A14.cif"]   # 다운로드 실패 시 커밋본 사용
CUTOFF = 4.0                                        # 본문 7.1 의 기본 cutoff

# ① 사슬 목록 먼저 — 어떤 사슬이 들어 있는지 눈으로 확인하고 나서 짝을 고릅니다.
run_tool([PY, SCRIPTS/"pdb_contacts.py", "--pdb", "1A14", "--outdir", "my_run/pdb", *FALLBACK])

# ② 짝별 contact 계산. 같은 CIF 를 재사용하니 두 번째부터는 [cache] 로 떠요.
PAIRS = [("H", "N", "항원-항체 — VH ↔ neuraminidase (본문이 보는 것)"),
         ("H", "L", "VH-VL packing — 같은 항체 내부 (대조군)")]

runs = []
for c1, c2, what in PAIRS:
    out = f"my_run/contacts_{c1}_{c2}.tsv"
    ok = run_tool([PY, SCRIPTS/"pdb_contacts.py", "--pdb", "1A14",
                   "--chain1", c1, "--chain2", c2, "--cutoff", str(CUTOFF),
                   "--outdir", "my_run/pdb", "--out", out, *FALLBACK])
    hits = [l for l in open(out)] if ok and pathlib.Path(out).exists() else []
    hits = [l for l in hits if "atom_contacts=" in l]
    runs.append({"짝": f"{c1}–{c2}", "무엇을 보는가": what, "잔기 pair": len(hits),
                 "atom contact": sum(int(l.rsplit("=", 1)[1]) for l in hits),
                 "결과 파일": out if hits else "실패 — 뒤 절은 data/ 로 이어갑니다"})

print(f"\ncutoff {CUTOFF} Å 로 계산한 두 짝")
display(pd.DataFrame(runs))
print("H–L 이 H–N 보다 pair 가 많은 건 정상이에요 — VH-VL 은 늘 맞물려 있는 내부 packing 이라,")
print("접촉이 많다고 결합이 강한 게 아닙니다. 우리가 볼 건 H–N 쪽이에요.")

## 2) 내 결과 확인 — paratope · epitope 집합과 CDR 귀속 (본문 7.3)

pair 개수만으로는 "어느 잔기가 붙었나" 를 말할 수 없어요. **paratope·epitope 잔기 집합**을 직접 뽑고,
paratope 이 정말 CDR 에 몰려 있는지 Kabat 구간으로 판정합니다.

In [ ]:
import pandas as pd
from IPython.display import display

KABAT_H = [("CDR-H1", 31, 35), ("CDR-H2", 50, 65), ("CDR-H3", 95, 102)]   # Kabat 정의

def parse_contacts(path):
    """pdb_contacts.py TSV → [(잔기1, 잔기2, atom_contacts)]. 잔기 라벨은 'ASN H:54' 꼴."""
    rows = []
    for line in open(path):
        if "atom_contacts=" not in line:
            continue
        left, n = line.rstrip().split("atom_contacts=")
        a, b = left.rstrip("\t").split("\t")
        rows.append((a.strip(), b.strip(), int(n)))
    return rows

def chain_of(label):
    return label.split()[-1].split(":")[0]          # 'ASN H:54' → 'H'

def resi_of(label):
    return label.split(":")[-1]                     # 'TYR H:100A' → '100A' (insertion code 포함)

def resi_key(resi):
    num = "".join(ch for ch in resi if ch.isdigit())
    return (int(num) if num else 0, "".join(ch for ch in resi if ch.isalpha()))

def load_contacts(path, left_role="paratope", right_role="epitope"):
    """역할 이름은 인자로 받고, 사슬 ID 는 파일에서 읽어 컬럼 라벨을 만든다."""
    rows = parse_contacts(path)
    c1 = chain_of(rows[0][0]) if rows else "?"
    c2 = chain_of(rows[0][1]) if rows else "?"
    return pd.DataFrame(rows, columns=[f"{left_role} ({c1})", f"{right_role} ({c2})", "atom_contacts"])

def residues(series):
    return sorted({resi_of(x) for x in series}, key=resi_key)

def cdr_of(resi):
    num = int("".join(ch for ch in resi if ch.isdigit()) or 0)
    for name, lo, hi in KABAT_H:
        if lo <= num <= hi:
            return name
    return "FR"

hn_path = resolve("contacts_H_N.tsv")
hn = load_contacts(hn_path).sort_values("atom_contacts", ascending=False).reset_index(drop=True)
pcol, ecol = hn.columns[0], hn.columns[1]
ab_chain, ag_chain = chain_of(hn[pcol].iloc[0]), chain_of(hn[ecol].iloc[0])

print(f"항원-항체 contact ({CUTOFF} Å) — {len(hn)} residue pairs · "
      f"총 {hn['atom_contacts'].sum()} atom contacts")
display(hn)                                          # 15행 전부 (잘라내지 않아요)

para, epi = residues(hn[pcol]), residues(hn[ecol])
print(f"paratope ({ab_chain}) {len(para)}개 —", " · ".join(ab_chain + r for r in para))
print(f"epitope  ({ag_chain}) {len(epi)}개 —", " · ".join(ag_chain + r for r in epi))

cdr_tab = pd.DataFrame({"paratope": [ab_chain + r for r in para],
                        "region": [cdr_of(r) for r in para],
                        "atom_contacts": [int(hn.loc[hn[pcol].map(resi_of) == r, "atom_contacts"].sum())
                                          for r in para]})
display(cdr_tab)
n_cdr = int((cdr_tab["region"] != "FR").sum())
vc = cdr_tab["region"].value_counts()
print("CDR 분포 —", " · ".join(f"{k} {v}개" for k, v in sorted(vc.items())))
if n_cdr == len(para):
    print(f"→ paratope {len(para)}개가 하나도 빠짐없이 CDR 안이에요. 이론대로 CDR 이 결합을 주도합니다.")
else:
    print(f"→ paratope {len(para)}개 중 CDR {n_cdr}개 · FR {len(para) - n_cdr}개. "
          f"FR 접촉 잔기는 humanization 때 보존 후보로 따로 표시하세요.")

hl = load_contacts(resolve("contacts_H_L.tsv"), "VH", "VL")
print(f"\n비교) 항원 H–{ag_chain} = {len(hn)} pairs / {hn['atom_contacts'].sum()} atom contacts "
      f"vs VH–VL packing = {len(hl)} pairs / {hl['atom_contacts'].sum()} atom contacts")
print("같은 구조·같은 cutoff 인데 '무엇 대 무엇'이냐로 결과가 완전히 달라져요 — "
      "보고서에는 chain 쌍을 반드시 적으세요.")

## 3) 내 결과 확인 — interface contact 그래프 (본문 7.3)

막대가 긴(원자 접촉이 많은) 잔기일수록 affinity maturation·humanization 에서 **함부로 바꾸면 안 되는
hot-spot** 이에요. 2절 표의 상위 행이 그대로 막대 위쪽에 옵니다.

In [ ]:
from antibody_viz import interface_contacts
from IPython.display import Image, display

png = "my_run/07_interface_contacts.png"
interface_contacts(hn_path,                          # 2절이 고른 그 TSV
                   f"Antibody({ab_chain})–Antigen({ag_chain}) contacts — 1A14 "
                   f"(≤{CUTOFF} Å, 내 실행 결과)", png)
display(Image(png))

top = hn.iloc[0]
print(f"가장 긴 막대 = {top[pcol]} ↔ {top[ecol]} ({top['atom_contacts']} atom contacts, "
      f"{cdr_of(resi_of(top[pcol]))})")
print(f"상위 3쌍이 전체 {hn['atom_contacts'].sum()} atom contacts 중 "
      f"{hn['atom_contacts'].head(3).sum()}개 — 접촉이 소수 잔기에 몰릴수록 그 잔기의 변이 위험이 커요.")

## 4) 레퍼런스 대조 — 커밋된 contact 결과와 같은가

내가 받은 CIF 로 계산한 pair 집합이 커밋된 `data/contacts_H_N.tsv` 와 같은지 봅니다.
같으면 다운로드·파싱·거리계산이 모두 재현된 거예요.

In [ ]:
import pathlib

mine_p = pathlib.Path("my_run/contacts_H_N.tsv")
if mine_p.exists():
    mine = {(a, b): n for a, b, n in parse_contacts(mine_p)}
    ref = {(a, b): n for a, b, n in parse_contacts("data/contacts_H_N.tsv")}
    print(f"내 결과 {len(mine)} pairs / 레퍼런스 {len(ref)} pairs")

    only_mine, only_ref = sorted(set(mine) - set(ref)), sorted(set(ref) - set(mine))
    diff_n = {k: (mine[k], ref[k]) for k in set(mine) & set(ref) if mine[k] != ref[k]}
    if only_mine or only_ref or diff_n:
        display(pd.DataFrame(
            [{"pair": f"{a} ↔ {b}", "차이": "내 결과에만 있음",
              "내 결과": mine[(a, b)], "레퍼런스": "-"} for a, b in only_mine]
            + [{"pair": f"{a} ↔ {b}", "차이": "레퍼런스에만 있음",
                "내 결과": "-", "레퍼런스": ref[(a, b)]} for a, b in only_ref]
            + [{"pair": f"{a} ↔ {b}", "차이": "원자접촉 수가 다름",
                "내 결과": m, "레퍼런스": r} for (a, b), (m, r) in sorted(diff_n.items())]))
    else:
        print("어긋난 pair 없음 — 빠진 것도, 더 생긴 것도, 접촉 수가 다른 것도 0개예요.")

    if mine == ref:
        print("→ pair 집합과 접촉 수가 완전히 일치. 재현 성공이에요.")
    else:
        print("→ 차이가 있어요. 같은 PDB entry·같은 cutoff 를 썼는지, "
              "구조 버전이 갱신되지 않았는지 확인하세요.")
else:
    print("my_run contact 결과가 없어 대조를 건너뜁니다.")

## 5) cutoff 를 바꾸면 얼마나 달라지나 (본문 7.1)

본문 7.1 이 말하는 대로 **4 Å 은 시작점**이에요. contact 수는 결합의 세기가 아니고,
H-bond geometry·buried surface area(BSA)·shape complementarity 는 FreeSASA·PLIP 같은
**별도 도구**의 몫입니다(본문 7.4). cutoff 를 0.5 Å 만 늘려도 결과가 얼마나 움직이는지 직접 재 봐요.

In [ ]:
CUT2 = 4.5
out2 = f"my_run/contacts_H_N_{CUT2}.tsv"
run_tool([PY, SCRIPTS/"pdb_contacts.py", "--pdb", "1A14",
          "--chain1", "H", "--chain2", ag_chain, "--cutoff", str(CUT2),
          "--outdir", "my_run/pdb", "--out", out2, *FALLBACK])

if pathlib.Path(out2).exists():
    wide = load_contacts(out2)
    e1, e2 = set(residues(hn[ecol])), set(residues(wide[wide.columns[1]]))
    p1, p2 = set(residues(hn[pcol])), set(residues(wide[wide.columns[0]]))
    n1, n2 = int(hn["atom_contacts"].sum()), int(wide["atom_contacts"].sum())
    print(f"\ncutoff 만 {CUTOFF} → {CUT2} Å 로 바꿔서 같은 짝(H–{ag_chain})을 다시 계산했어요.")
    display(pd.DataFrame([
        {"cutoff (Å)": CUTOFF, "잔기 pair": len(hn), "atom contact": n1,
         "epitope 잔기": len(e1), "paratope 잔기": len(p1)},
        {"cutoff (Å)": CUT2, "잔기 pair": len(wide), "atom contact": n2,
         "epitope 잔기": len(e2), "paratope 잔기": len(p2)},
    ]))
    print("0.5 Å 늘렸을 때 새로 들어온 잔기")
    print("  epitope  — " + (" · ".join(sorted(e2 - e1, key=resi_key)) or "없음"))
    print("  paratope — " + (" · ".join(sorted(p2 - p1, key=resi_key)) or "없음"))
    print(f"→ 0.5 Å 차이로 pair 가 {len(hn)}→{len(wide)}, atom contact 가 {n1}→{n2} 로 움직여요.")
    print("   epitope 목록을 인용할 때는 cutoff 를 반드시 함께 적어야 비교가 성립합니다.")
else:
    print(f"{CUT2} Å 재계산 결과가 없어 이 절을 건너뜁니다.")

## 6) 복합체 3D 렌더 — paratope / epitope 인라인 뷰어 (본문 7.3)

**내가 받은 그 구조를 노트북 안에서 바로 돌려 봅니다**(py3Dmol, Colab 그대로 동작).

- cartoon 색 — 항원(베이지) · 항체 H(하늘) · L(연두)
- **주황 스틱 = 2절이 계산한 paratope · 빨간 스틱 = 2절이 계산한 epitope.**
  잔기 번호를 여기 적어 두지 않고 앞 셀의 `para`·`epi` 변수를 그대로 씁니다 —
  cutoff 를 바꿔 다시 돌리면 **그림도 따라 바뀌어요**(예전 `.pml` 은 고정 selection 이라 계산 결과와 어긋났어요).
- 강조는 **별도 모델로 얹습니다** — `H100A`·`H100B` 처럼 insertion code 가 붙은 잔기가
  `resi` 선택에서 새는 걸 원천 차단하고, 아래에서 "못 찾은 잔기 0개"까지 확인합니다.

뷰어는 두 개예요 — **① 복합체 전체**(어느 사슬이 어디 붙었는지) **② 인터페이스 확대**(스틱이 맞물리는 모습).

In [ ]:
import importlib.util, sys, pathlib
for _pkg in ("py3Dmol", "gemmi"):        # 부트스트랩이 이미 깔지만, 이 셀만 따로 돌릴 때 대비
    if importlib.util.find_spec(_pkg) is None:
        _run(f'"{sys.executable}" -m pip -q install {_pkg}')
import py3Dmol, gemmi

cif = pathlib.Path(resolve("pdb/1A14.cif"))          # my_run 우선, 없으면 커밋본
assert cif.exists(), f"{cif} 가 없어요 — 1절을 먼저 실행하세요."
WHICH = "내 결과 (my_run/)" if cif.parts[0] == "my_run" else "레퍼런스 (data/ 커밋본)"
print(f"[3D 뷰어] 표시 대상 = {WHICH} — {cif}")

st = gemmi.read_structure(str(cif)); st.setup_entities()
cx = st.make_pdb_string()                            # CIF → PDB 문자열 (3Dmol.js 가 가장 안정적)

def keys_of(chain, resis):
    """['52','100A',...] → {(chain, 52, ''), (chain, 100, 'A'), ...}  insertion code 보존"""
    out = set()
    for r in resis:
        out.add((chain, int("".join(c for c in r if c.isdigit()) or 0),
                 "".join(c for c in r if c.isalpha())))
    return out

def sub_pdb(text, keys):
    return "".join(L for L in text.splitlines(keepends=True)
                   if L.startswith(("ATOM", "HETATM"))
                   and (L[21], int(L[22:26]), L[26].strip()) in keys)

para_keys, epi_keys = keys_of(ab_chain, para), keys_of(ag_chain, epi)   # 2절 계산 결과 그대로
para_pdb,  epi_pdb  = sub_pdb(cx, para_keys), sub_pdb(cx, epi_keys)
others = sorted({L[21] for L in cx.splitlines() if L.startswith("ATOM")} - {ab_chain, ag_chain})

def build(width=860, height=580):
    """복합체 cartoon + paratope/epitope 스틱. 반환값 = (view, 강조 모델 인덱스)"""
    v = py3Dmol.view(width=width, height=height)
    v.addModel(cx, "pdb")
    v.setStyle({"model": -1}, {"cartoon": {"color": "lightgrey"}})
    v.setStyle({"model": -1, "chain": ag_chain}, {"cartoon": {"color": "wheat"}})
    v.setStyle({"model": -1, "chain": ab_chain}, {"cartoon": {"color": "skyblue"}})
    for ch in others:
        v.setStyle({"model": -1, "chain": ch}, {"cartoon": {"color": "palegreen"}})
    idx, n = [], 1
    for text, scheme in ((epi_pdb, "redCarbon"), (para_pdb, "orangeCarbon")):
        if text:
            v.addModel(text, "pdb")
            v.setStyle({"model": -1}, {"stick": {"radius": 0.28, "colorscheme": scheme}})
            idx.append(n); n += 1
    v.setBackgroundColor("white")
    return v, idx

v1, hl = build()
v1.zoomTo()                                          # ① 복합체 전체
v1.show()

v2, hl2 = build(height=520)
v2.zoomTo({"model": hl2} if hl2 else {})             # ② 인터페이스 확대 (강조 모델에 맞춤)
v2.show()

print(f"사슬 색 — 항원 {ag_chain}(베이지) · 항체 {ab_chain}(하늘) · {others}(연두)")
print(f"주황 스틱 paratope {len(para)}개 — " + " · ".join(ab_chain + r for r in para))
print(f"빨간 스틱 epitope  {len(epi)}개 — " + " · ".join(ag_chain + r for r in epi))
found = {(L[21], int(L[22:26]), L[26].strip()) for L in (para_pdb + epi_pdb).splitlines()}
missing = (para_keys | epi_keys) - found
print(f"강조에 들어간 원자 = paratope {len(para_pdb.splitlines())}개 · epitope {len(epi_pdb.splitlines())}개 | "
      f"좌표에서 못 찾은 잔기 = {[f'{c}{n}{i}' for c, n, i in sorted(missing)] or '없음'}")
print("\n판정 — ① 주황 스틱이 빨간 패치로 파고드는가 ② 빨간 잔기가 항원 표면 한 자리에 모여 있는가"
      "(모여 있으면 conformational epitope) ③ 위 목록이 2절 표의 잔기와 같은가.")

## 6b) (선택) PyMOL 고해상도 정지 이미지

**PyMOL 은 pip 로 설치되지 않아요**(Colab 미지원) — 6절 인라인 뷰어가 주 시각화이고 이 절은 덤이에요.
PyMOL 이 있으면 내가 받은 CIF 로 다시 렌더해 `my_run/07_complex_3d.png` 로 저장하고,
없으면 커밋된 렌더를 **레퍼런스라고 밝히고** 보여줍니다.
커밋된 `scripts/render_07_complex.pml` 은 건드리지 않고 **경로·selection 만 바꾼 사본**을 써요.

In [ ]:
import shutil, subprocess

def repath_pml(src, load_path, png_path, selections=None):
    """커밋된 .pml 의 load/png 경로와 select 문만 바꾼 사본 텍스트를 만든다."""
    out = []
    for line in pathlib.Path(src).read_text().splitlines():
        s = line.strip()
        rest = ("," + line.split(",", 1)[1]) if "," in line else ""
        if s.startswith("load "):
            line = f"load {load_path}{rest}"
        elif s.startswith("png "):
            line = f"png {png_path}{rest}"
        elif selections and s.startswith("select "):
            key = s.split(",", 1)[0].split()[1]
            if key in selections:
                line = f"select {key}, {selections[key]}"
        out.append(line)
    return "\n".join(out) + "\n"

committed = "07_complex_3d.png"                   # 커밋본 (덮어쓰지 않아요)
mine_png = (MYRUN/"07_complex_3d.png").resolve()
shown, origin = committed, "레퍼런스(커밋된 렌더) — 내 계산 결과가 아닙니다"
note = "PyMOL 없음(예: Colab) → 커밋된 렌더를 표시합니다. 내 결과는 6절 뷰어에서 보세요."

if shutil.which("pymol"):
    sel = {"para": f"chain {ab_chain} and resi " + "+".join(para),
           "epi":  f"chain {ag_chain} and resi " + "+".join(epi)}
    tmp_pml = MYRUN/"render_07_complex.my_run.pml"
    tmp_pml.write_text(repath_pml(ADV_ROOT/"scripts"/"render_07_complex.pml",
                                  cif.resolve(), mine_png, selections=sel))   # 6절이 고른 그 CIF
    print("selection —", sel)
    try:
        subprocess.run(["pymol", "-cq", str(tmp_pml)], cwd=str(ADV_ROOT),
                       check=True, capture_output=True, text=True, timeout=180)
        shown, origin = str(mine_png), f"내 계산 결과로 재렌더 ({cif})"
        note = f"PyMOL 재렌더 완료 → {mine_png}"
    except Exception as e:
        note = f"PyMOL 실행 실패({type(e).__name__}) → 커밋된 렌더를 표시합니다."

print(note)
display(Image(shown))
print(f"판정 — 표시한 이미지 = {shown}  |  출처 = {origin}")

> 다음 → 본문 [08. developability](../08_developability/08_developability.md)